### делаем 2 коллекции

In [19]:
from platform import python_version
import os
import re
import uuid
import json
from collections import defaultdict

import pandas as pd
from dotenv import load_dotenv

from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

import clickhouse_connect

load_dotenv()



True

In [20]:
from langchain_ollama import ChatOllama

In [21]:
CH_HOST = os.getenv("CH_HOST")
CH_PORT = int(os.getenv("CH_PORT"))
CLICKHOUSE_USER = os.getenv("CH_USER")
CLICKHOUSE_PASSWORD = os.getenv("CH_PASSWORD")

In [22]:
CLICKHOUSE_PASSWORD

'1234'

In [23]:
# ---------------------------------------------------
# 1. Подключения
# ---------------------------------------------------

QDRANT_URL = os.getenv("QDRANT_URL")
client_qdrant = QdrantClient(url=QDRANT_URL)

CH_HOST = os.getenv("CH_HOST")
CH_PORT = int(os.getenv("CH_PORT"))
CLICKHOUSE_USER = os.getenv("CH_USER")
CLICKHOUSE_PASSWORD = os.getenv("CH_PASSWORD")

client_clickhouse = clickhouse_connect.get_client(
    host=CH_HOST,
    port=CH_PORT,
    username=CLICKHOUSE_USER,
    password=CLICKHOUSE_PASSWORD
)

# BASE_URL = os.getenv("EMBEDDINGS_BASE_URL", "http://localhost:8000/v1")
#
# embeddings = OpenAIEmbeddings(
#     model="Qwen/Qwen3-Embedding-0.6B",
#     api_key="not-needed",
#     base_url=BASE_URL,
#     tiktoken_enabled=False,
# )
#
# vec = embeddings.embed_query("test")
# EMBEDDING_DIM = len(vec)
#
# print("Python:", python_version())
# print("Embedding dim:", EMBEDDING_DIM)



In [6]:
# ---------------------------------------------------
# 2. Имена коллекций
# ---------------------------------------------------

MESSAGES_COLLECTION = "mailkb_messages"
THREADS_COLLECTION = "mailkb_threads"


# ---------------------------------------------------
# 3. Настройка коллекций
# ---------------------------------------------------

def ensure_collection(collection_name: str, recreate: bool = False) -> QdrantVectorStore:
    collections = client_qdrant.get_collections().collections
    existing_names = {c.name for c in collections}

    if recreate and collection_name in existing_names:
        client_qdrant.delete_collection(collection_name=collection_name)
        existing_names.remove(collection_name)

    if collection_name not in existing_names:
        client_qdrant.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(
                size=EMBEDDING_DIM,
                distance=Distance.COSINE,
            ),
        )

    return QdrantVectorStore(
        client=client_qdrant,
        collection_name=collection_name,
        embedding=embeddings,
    )


# recreate=True только если хочешь пересобрать с нуля
messages_qv = ensure_collection(MESSAGES_COLLECTION, recreate=False)
threads_qv = ensure_collection(THREADS_COLLECTION, recreate=False)



In [24]:
# ---------------------------------------------------
# 4. Утилиты
# ---------------------------------------------------

RE_PREFIX = re.compile(r'^\s*(re|fw|fwd|aw|ответ):\s*', flags=re.IGNORECASE)

def normalize_subject(subj: str) -> str:
    s = (subj or "").strip()
    while True:
        ns = RE_PREFIX.sub('', s).strip()
        if ns == s:
            break
        s = ns
    s = re.sub(r'\s+', ' ', s)
    return s.lower()

def split_addrs(x) -> list[str]:
    if x is None:
        return []
    if isinstance(x, list):
        return [str(v).strip() for v in x if str(v).strip()]
    return [p.strip() for p in re.split(r'[;,]', str(x)) if p.strip()]

def participants_list(row) -> list[str]:
    people = (
        split_addrs(row.get("from_addr")) +
        split_addrs(row.get("to_addr")) +
        split_addrs(row.get("cc_addr")) +
        split_addrs(row.get("bcc_addr"))
    )
    return sorted(set(people))

def safe_json_loads(value):
    if value is None:
        return None
    if isinstance(value, dict):
        return value
    if isinstance(value, str):
        try:
            return json.loads(value)
        except Exception:
            return None
    return None

# ---------------------------------------------------
# 5. SQL: распарсенные письма из emails_unique
# ---------------------------------------------------

def load_join_batch(limit: int, offset: int) -> pd.DataFrame:
    query = f"""
    SELECT
        e.id,
        e.thread_key,
        e.message_id,
        e.subject,
        e.from_addr,
        e.to_addr,
        e.cc_addr,
        e.bcc_addr,
        e.sent_at_utc,
        e.folder,
        p.parsed_json
    FROM mailkb.emails_unique e
    INNER JOIN mailkb.mail_parsed p
        ON e.id = p.email_id
    ORDER BY e.sent_at_utc ASC, e.id ASC
    LIMIT {limit} OFFSET {offset}
    """
    return client_clickhouse.query_df(query)

def load_all_joined_df() -> pd.DataFrame:
    query = """
    SELECT
        e.id,
        e.thread_key,
        e.message_id,
        e.subject,
        e.from_addr,
        e.to_addr,
        e.cc_addr,
        e.bcc_addr,
        e.sent_at_utc,
        e.folder,
        p.parsed_json
    FROM mailkb.emails_unique e
    INNER JOIN mailkb.mail_parsed p
        ON e.id = p.email_id
    ORDER BY e.sent_at_utc ASC, e.id ASC
    """
    return client_clickhouse.query_df(query)


In [25]:
# ---------------------------------------------------
# 4. Утилиты
# ---------------------------------------------------

RE_PREFIX = re.compile(r'^\s*(re|fw|fwd|aw|ответ):\s*', flags=re.IGNORECASE)

def normalize_subject(subj: str) -> str:
    s = (subj or "").strip()
    while True:
        ns = RE_PREFIX.sub('', s).strip()
        if ns == s:
            break
        s = ns
    s = re.sub(r'\s+', ' ', s)
    return s.lower()

def split_addrs(x) -> list[str]:
    if x is None:
        return []
    if isinstance(x, list):
        return [str(v).strip() for v in x if str(v).strip()]
    return [p.strip() for p in re.split(r'[;,]', str(x)) if p.strip()]

def participants_list(row) -> list[str]:
    people = (
        split_addrs(row.get("from_addr")) +
        split_addrs(row.get("to_addr")) +
        split_addrs(row.get("cc_addr")) +
        split_addrs(row.get("bcc_addr"))
    )
    return sorted(set(people))

def safe_json_loads(value):
    if value is None:
        return None
    if isinstance(value, dict):
        return value
    if isinstance(value, str):
        try:
            return json.loads(value)
        except Exception:
            return None
    return None

# ---------------------------------------------------
# 5. SQL: распарсенные письма из emails_unique
# ---------------------------------------------------

def load_join_batch(limit: int, offset: int) -> pd.DataFrame:
    query = f"""
    SELECT
        e.id,
        e.thread_key,
        e.message_id,
        e.subject,
        e.from_addr,
        e.to_addr,
        e.cc_addr,
        e.bcc_addr,
        e.sent_at_utc,
        e.folder,
        p.parsed_json
    FROM mailkb.emails_unique e
    INNER JOIN mailkb.mail_parsed p
        ON e.id = p.email_id
    ORDER BY e.sent_at_utc ASC, e.id ASC
    LIMIT {limit} OFFSET {offset}
    """
    return client_clickhouse.query_df(query)

def load_all_joined_df() -> pd.DataFrame:
    query = """
    SELECT
        e.id,
        e.thread_key,
        e.message_id,
        e.subject,
        e.from_addr,
        e.to_addr,
        e.cc_addr,
        e.bcc_addr,
        e.sent_at_utc,
        e.folder,
        p.parsed_json
    FROM mailkb.emails_unique e
    INNER JOIN mailkb.mail_parsed p
        ON e.id = p.email_id
    ORDER BY e.sent_at_utc ASC, e.id ASC
    """
    return client_clickhouse.query_df(query)


In [26]:
# ---------------------------------------------------
# 6. Построение документов для messages
# ---------------------------------------------------

def build_message_docs(df: pd.DataFrame) -> list[Document]:
    docs: list[Document] = []

    for row in df.to_dict("records"):
        parsed = safe_json_loads(row.get("parsed_json"))
        if not parsed:
            continue

        parsed_emails = parsed.get("emails", [])
        if not isinstance(parsed_emails, list):
            continue

        subj = (row.get("subject") or "").strip()
        participants = participants_list(row)

        for msg in parsed_emails:
            if not isinstance(msg, dict):
                continue

            email_body = (msg.get("email_body") or "").strip()
            if not email_body:
                continue

            topic = (msg.get("topic") or subj or "").strip()
            msg_date = msg.get("date")
            mail_query_number = msg.get("mail_query_number")
            keywords = msg.get("key_words") or []
            thread_key = row.get("thread_key") or normalize_subject(subj)

            meta = {
                "email_id": row.get("id"),
                "message_id": row.get("message_id"),
                "subject": subj,
                "topic": topic,
                "date": str(msg_date) if msg_date is not None else str(row.get("sent_at_utc")),
                "mail_query_number": int(mail_query_number) if str(mail_query_number).isdigit() else mail_query_number,
                "keywords": keywords,
                "sent_at_utc": str(row.get("sent_at_utc")),
                "folder": row.get("folder"),
                "participants": participants,
                "thread_key": thread_key,
                "source_type": "message",
            }

            docs.append(Document(page_content=email_body, metadata=meta))

    return docs

def make_message_ids(docs: list[Document]) -> list[str]:
    ids = []
    for d in docs:
        raw = (
            f"{d.metadata.get('email_id')}::"
            f"{d.metadata.get('mail_query_number')}::"
            f"{d.metadata.get('date')}::"
            f"{d.page_content[:100]}"
        )
        uid = uuid.uuid5(uuid.NAMESPACE_URL, raw)
        ids.append(str(uid))
    return ids

def upload_message_docs(docs: list[Document], qv: QdrantVectorStore):
    if not docs:
        return 0

    ids = make_message_ids(docs)
    texts = [d.page_content for d in docs]
    metadatas = [d.metadata for d in docs]

    qv.add_texts(
        texts=texts,
        metadatas=metadatas,
        ids=ids
    )
    return len(docs)

def ingest_messages(batch_size: int = 1000):
    offset = 0
    total_docs = 0
    total_rows = 0

    while True:
        df_batch = load_join_batch(limit=batch_size, offset=offset)
        if df_batch.empty:
            break

        docs = build_message_docs(df_batch)
        inserted = upload_message_docs(docs, messages_qv)

        total_docs += inserted
        total_rows += len(df_batch)

        print(
            f"[messages] rows_processed={total_rows}, "
            f"docs_inserted_total={total_docs}, "
            f"last_batch_rows={len(df_batch)}, "
            f"last_batch_docs={inserted}"
        )

        offset += len(df_batch)

    print(f"[messages] DONE: rows_processed={total_rows}, docs_inserted_total={total_docs}")

In [8]:
# ---------------------------------------------------
# 8. Запуск
# ---------------------------------------------------

# если нужно пересобрать коллекции с нуля, поставь recreate=True выше
ingest_messages(batch_size=1000)
# ingest_threads()

[messages] rows_processed=460, docs_inserted_total=2332, last_batch_rows=460, last_batch_docs=2332
[messages] DONE: rows_processed=460, docs_inserted_total=2332


In [10]:
df_test = load_all_joined_df()
print(df_test.columns.tolist())
print(df_test[["id", "thread_key", "subject"]].head())

['id', 'thread_key', 'message_id', 'subject', 'from_addr', 'to_addr', 'cc_addr', 'bcc_addr', 'sent_at_utc', 'folder', 'parsed_json']
                                     id  \
0  609828c8-9f43-48a0-a0e0-a39b18694835   
1  874e7ad5-4709-4e50-87de-0783f75c0efe   
2  3b62442c-4b4f-4270-b3be-0151c0bb09b2   
3  73f9bdf9-8439-42a2-8ce7-02d5c70e1a05   
4  316dae1a-1704-463f-943a-6773055cfdbe   

                                          thread_key  \
0                        Список НД для регламента ОП   
1  Плановая маржинальность в sap Коллеги, мы вас ...   
2                                Закрытие инцидентов   
3                    ТМ Закрытие инцидентов - solman   
4  new time proposed: Обсуждение интеграции ЭТРАН...   

                                             subject  
0                        Список НД для регламента ОП  
1  RE: Плановая маржинальность в SAP Коллеги, мы ...  
2                            RE: Закрытие инцидентов  
3                    ТМ Закрытие инцидентов - solma

### агент

In [7]:
import json
from collections import defaultdict
from langchain.tools import tool
from collections import defaultdict
from pathlib import Path

from langchain.tools import tool

SUMMARY_DIR = Path("../../summaries")
SUMMARY_DIR.mkdir(exist_ok=True)

In [8]:
@tool
def search_project_threads(project_hint: str, limit: int = 30) -> str:
    """
    Найти релевантные обсуждения проекта в коллекции mailkb_threads.
    """
    docs = threads_qv.similarity_search(project_hint, k=limit)

    results = []
    seen = set()

    for doc in docs:
        md = doc.metadata or {}
        thread_key = md.get("thread_key")

        if not thread_key or thread_key in seen:
            continue
        seen.add(thread_key)

        results.append({
            "thread_key": thread_key,
            "subject": md.get("subject"),
            "participants": md.get("participants") or [],
            "keywords": md.get("keywords") or [],
            "message_count": md.get("message_count"),
            "first_date": md.get("first_date"),
            "last_date": md.get("last_date"),
            "snippet": (doc.page_content[:800] + "…") if doc.page_content else None,
        })

    return json.dumps(
        {"project_hint": project_hint, "threads": results},
        ensure_ascii=False,
        indent=2
    )


def _escape_ch_string(value: str) -> str:
    return value.replace("\\", "\\\\").replace("'", "\\'")


@tool
def get_project_corpus_batch(
        project_hint: str,
        offset: int = 0,
        batch_size: int = 50,
        thread_limit: int = 30
) -> str:
    """
    Собрать батч корпуса проекта:
    1) найти релевантные thread_key в mailkb_threads
    2) достать все сообщения по этим thread_key
    3) дедуплицировать и отсортировать
    4) вернуть батч по offset/batch_size
    """
    thread_docs = threads_qv.similarity_search(project_hint, k=thread_limit)

    thread_keys = []
    seen = set()

    for doc in thread_docs:
        md = doc.metadata or {}
        tk = md.get("thread_key")
        if tk and tk not in seen:
            seen.add(tk)
            thread_keys.append(tk)

    if not thread_keys:
        return json.dumps({
            "project_hint": project_hint,
            "offset": offset,
            "batch_size": batch_size,
            "batch_len": 0,
            "has_more": False,
            "thread_keys": [],
            "total_messages": 0,
            "batch": []
        }, ensure_ascii=False, indent=2)

    quoted_keys = ", ".join(f"'{_escape_ch_string(tk)}'" for tk in thread_keys)

    query = f"""
    SELECT
        e.id,
        e.thread_key,
        e.message_id,
        e.subject,
        e.from_addr,
        e.to_addr,
        e.cc_addr,
        e.bcc_addr,
        e.sent_at_utc,
        e.folder,
        p.parsed_json
    FROM mailkb.emails_unique e
    INNER JOIN mailkb.mail_parsed p
        ON e.id = p.email_id
    WHERE e.thread_key IN ({quoted_keys})
    ORDER BY e.sent_at_utc ASC, e.id ASC
    """

    df = client_clickhouse.query_df(query)
    docs = build_message_docs(df)

    deduped = []
    seen_msg = set()

    for d in docs:
        dedup_key = (
            f"{d.metadata.get('email_id')}::"
            f"{d.metadata.get('mail_query_number')}::"
            f"{d.metadata.get('date')}::"
            f"{d.page_content[:120]}"
        )
        if dedup_key in seen_msg:
            continue
        seen_msg.add(dedup_key)
        deduped.append(d)

    deduped.sort(
        key=lambda d: (
            d.metadata.get("date") or d.metadata.get("sent_at_utc") or "",
            str(d.metadata.get("mail_query_number") or "")
        )
    )

    batch_docs = deduped[offset: offset + batch_size]

    batch = []
    for d in batch_docs:
        md = d.metadata or {}
        batch.append({
            "thread_key": md.get("thread_key"),
            "email_id": md.get("email_id"),
            "message_id": md.get("message_id"),
            "subject": md.get("subject"),
            "topic": md.get("topic"),
            "sent_at_utc": md.get("date") or md.get("sent_at_utc"),
            "participants": md.get("participants") or [],
            "keywords": md.get("keywords") or [],
            "folder": md.get("folder"),
            "snippet": d.page_content[:800] if d.page_content else None,
        })

    return json.dumps({
        "project_hint": project_hint,
        "offset": offset,
        "batch_size": batch_size,
        "batch_len": len(batch),
        "has_more": offset + batch_size < len(deduped),
        "thread_keys": thread_keys,
        "total_messages": len(deduped),
        "batch": batch
    }, ensure_ascii=False, indent=2)


@tool
def save_summary(summary_text: str, batch_id: int) -> str:
    """
    Сохранить summary батча в summaries/summary_batch_{batch_id}.txt
    """
    p = SUMMARY_DIR / f"summary_batch_{batch_id}.txt"
    p.write_text(summary_text, encoding="utf-8")
    return f"saved_to={str(p)}"


@tool
def load_all_summaries() -> str:
    """
    Прочитать все summary_batch_*.txt и вернуть объединённый текст.
    """
    files = sorted(SUMMARY_DIR.glob("summary_batch_*.txt"))

    if not files:
        return "NO_SUMMARIES_FOUND"

    texts = []
    for file in files:
        content = file.read_text(encoding="utf-8")
        texts.append(f"===== {file.name} =====\n{content}\n")

    return "\n".join(texts)

In [13]:
print(search_project_threads.invoke({"project_hint": "Segezha", "limit": 10}))

{
  "project_hint": "Segezha",
  "threads": [
    {
      "thread_key": "Обсуждение вопросов по ЗНИ-282 \"ЗНИ_Изменение схемы учета вывозки сырья с филиалов Лесных ресурсов на СЦБК\"",
      "subject": "RE: Обсуждение вопросов по ЗНИ-282 \"ЗНИ_Изменение схемы учета вывозки сырья с филиалов Лесных ресурсов на СЦБК\"",
      "participants": [
        "IMCEAEX-_o=ilp_ou=Exchange+20Administrative+20Group+20+28FYDIBOHF23SPDLT+29_cn=Recipients_cn=user128488d2@bearingpoint.ru",
        "Khusainov_DS@segezha-group.com",
        "Lytkin_AV@segezha-group.com",
        "Safonov_AY@segezha-group.com",
        "alexey.ivanenko@bearingpoint.ru",
        "andrey.kryukov@bearingpoint.ru",
        "ce-sergey.schepin@bearingpoint.ru",
        "dmitry.kubyshkin@bearingpoint.ru",
        "petr.ostrik@bearingpoint.ru",
        "rasul.abdulin@bearingpoint.ru"
      ],
      "keywords": [
        "Skype",
        "Алексей",
        "Алексей Лыткин",
        "ЗНИ",
        "ЗНИ MNF",
        "ЗНИ с номером",


In [20]:
print(get_project_corpus_batch.invoke({
    "project_hint": "Segezha",
    "offset": 0,
    "batch_size": 5,
    "thread_limit": 2 # было 10
}))

{
  "project_hint": "Segezha",
  "offset": 0,
  "batch_size": 5,
  "batch_len": 5,
  "has_more": true,
  "thread_keys": [
    "Интеграция ТМ с ЭТРАН на пилотной площадке ВФК"
  ],
  "total_messages": 132,
  "batch": [
    {
      "thread_key": "Интеграция ТМ с ЭТРАН на пилотной площадке ВФК",
      "email_id": "ef1f8452-7806-4b72-80e6-dc3fac2303e7",
      "message_id": "<bd13504b14f44d3c94fc4835eea77762@ex2.be.local>",
      "subject": "FW:  Интеграция ТМ с ЭТРАН на пилотной площадке ВФК",
      "topic": "Интеграция SAP TM с ЭТРАН",
      "sent_at_utc": "2020-11-25",
      "participants": [
        "ce-Alexey.Fedoseev@bearingpoint.ru",
        "ce-sergey.schepin@bearingpoint.ru",
        "olga.alekseeva@bearingpoint.ru",
        "petr.ostrik@bearingpoint.ru"
      ],
      "keywords": [
        "интеграция",
        "SAP TM",
        "ЭТРАН",
        "график",
        "ВФК",
        "тиражирование"
      ],
      "folder": "Архивы\\Входящие",
      "snippet": "Коллеги, добрый день!\n\n

###  1. Batch-agent

In [11]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware, SummarizationMiddleware

SYSTEM_PROMPT_BATCH = """
Ты — аналитик проектной переписки.

Твоя задача — обработать корпус сообщений по проекту батчами.
Корпус уже собран из релевантных обсуждений проекта.

Работай строго по циклу:

1. Определи project_hint из запроса пользователя.
2. Вызови get_project_corpus_batch(project_hint, offset, batch_size=50).
3. Получи batch.
4. На основе только этого batch сделай summary.
5. Сохрани summary через save_summary(summary_text, batch_id).
6. Если has_more = true — увеличь offset на batch_size и продолжай.
7. Если has_more = false — сообщи, что все батчи обработаны.

Ограничения:
- Не делай глобального отчёта.
- Не объединяй выводы между батчами.
- Каждый summary должен описывать только текущий batch.
- Не пропускай save_summary.
- Не используй никакие другие инструменты кроме get_project_corpus_batch и save_summary.

Для каждого батча извлекай:
1. Основные темы
2. Что обсуждалось
3. Решения / договорённости
4. Проблемы / риски / открытые вопросы
5. Явные задачи
6. Вероятные задачи
7. Явных ответственных
8. Вероятных ответственных
9. Краткий вывод по батчу

Важно:
- Если задача или ответственный не сформулированы явно, можешь аккуратно реконструировать их,
  но обязательно явно разделяй:
  - "Явно указано"
  - "Вероятно следует из контекста"
"""

# agent_batch = create_agent(
#     model="deepseek-chat",
#     tools=[
#         get_project_corpus_batch,
#         save_summary,
#     ],
#     system_prompt=SYSTEM_PROMPT_BATCH,
#     middleware=[
#         PIIMiddleware("email", strategy="redact", apply_to_input=True),
#         PIIMiddleware(
#             "phone_number",
#             detector=(
#                 r"(?:\+?\d{1,3}[\s.-]?)?"
#                 r"(?:\(?\d{2,4}\)?[\s.-]?)?"
#                 r"\d{3,4}[\s.-]?\d{4}"
#             ),
#             strategy="redact",
#         ),
#         SummarizationMiddleware(
#             model="deepseek-chat",
#             max_tokens_before_summary=1200,
#         ),
#     ],
# )
model_local = ChatOllama(
    model="gemma4:26b",
    base_url="http://localhost:11434",
    temperature=0,
)


agent_batch = create_agent(
    model= model_local,
    tools=[
        get_project_corpus_batch,
        save_summary,
    ],
    system_prompt=SYSTEM_PROMPT_BATCH,
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware(
            "phone_number",
            detector=(
                r"(?:\+?\d{1,3}[\s.-]?)?"
                r"(?:\(?\d{2,4}\)?[\s.-]?)?"
                r"\d{3,4}[\s.-]?\d{4}"
            ),
            strategy="redact",
        ),
        SummarizationMiddleware(
            model="deepseek-chat",
            max_tokens_before_summary=1200,
        ),
    ],
)

In [9]:
result_batch = agent_batch.invoke({
    "messages": [
        {
            "role": "user",
            "content": (
                "Начни обработку проекта Segezha батчами. "
                "Делай summary каждого батча и сохраняй их через save_summary. "
                "Итоговый отчёт не делай."
            )
        }
    ]
})

print(result_batch["messages"][-1].content)

ResponseError: model 'gemma4:26b' not found (status code: 404)

### 2. Global-agent

In [9]:
from langchain_ollama import ChatOllama


In [30]:
SYSTEM_PROMPT_GLOBAL = """
Ты — эксперт по агрегированию проектной переписки.

Твоя задача:
1. Вызвать load_all_summaries().
2. На основе всех summary батчей сделать единый итоговый отчёт по проекту.

Структура итогового отчёта:
1. Краткое резюме проекта
2. Основные темы обсуждений
3. Ключевые решения и к чему пришли
4. Явные задачи и ответственные
5. Вероятные задачи и вероятные ответственные
6. Проблемы, риски, открытые вопросы
7. Повторяющиеся инциденты / узкие места
8. Итоговый вывод

Важно:
- Не переписывай summaries батчей дословно.
- Делай смысловую агрегацию.
- Если задача/ответственный восстановлены по контексту, помечай это отдельно.
- Если есть противоречия между батчами, укажи их.
"""

# agent_global = create_agent(
#     model="deepseek-chat",
#     tools=[load_all_summaries],
#     system_prompt=SYSTEM_PROMPT_GLOBAL,
#     middleware=[
#         PIIMiddleware("email", strategy="redact", apply_to_input=True),
#         PIIMiddleware(
#             "phone_number",
#             detector=(
#                 r"(?:\+?\d{1,3}[\s.-]?)?"
#                 r"(?:\(?\d{2,4}\)?[\s.-]?)?"
#                 r"\d{3,4}[\s.-]?\d{4}"
#             ),
#             strategy="redact",
#         ),
#         SummarizationMiddleware(
#             model="deepseek-chat",
#             max_tokens_before_summary=2000,
#         ),
#     ],
# )

model_local = ChatOllama(
    model="gemma4:26b",
    base_url="http://localhost:11434",
    temperature=0,
)
agent_global = create_agent(
    model=model_local,
    tools=[load_all_summaries],
    system_prompt=SYSTEM_PROMPT_GLOBAL,
    # middleware=[
    #     PIIMiddleware("email", strategy="redact", apply_to_input=True),
    #     PIIMiddleware(
    #         "phone_number",
    #         detector=(
    #             r"(?:\+?\d{1,3}[\s.-]?)?"
    #             r"(?:\(?\d{2,4}\)?[\s.-]?)?"
    #             r"\d{3,4}[\s.-]?\d{4}"
    #         ),
    #         strategy="redact",
    #     ),
    #     SummarizationMiddleware(
    #         model="deepseek-chat",
    #         max_tokens_before_summary=200,
    #     ),
    # ],
)

In [32]:
result_global = agent_global.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Сделай итоговый отчёт по проекту Segezha на основе сохранённых batch summaries."
        }
    ]
})

print(result_global["messages"][-1].content)

На основе предоставленного резюме (Batch 1) подготовлен итоговый отчёт по проекту интеграции SAP TM с системой ЭТРАН.

# Итоговый отчёт по проекту Segezha (Интеграция SAP TM — ЭТРАН)

### 1. Краткое резюме проекта
Проект направлен на настройку бесшовной интеграции между системой управления транспортной логистикой SAP TM и системой ЭТРАН. Основная цель текущего этапа — выбор пилотной площадки для запуска первого сценария интеграции («Загрузка Заявки — передача накладной») и последующее тиражирование решения на остальные производственные площадки компании.

### 2. Основные темы обсуждений
* **Выбор пилотной площадки:** Сравнение двух вариантов — ВФК и СЦБК.
* **Сценарии интеграции:** Определение приоритетных бизнес-процессов для первого этапа (заявки и накладные).
* **Техническая реализация:** Требования к сетевой связности (VipNet), режиму подключения (АСУ-АСУ) и необходимой документации.
* **Стратегия масштабирования:** План перехода от пилота к полноценному внедрению на всех площадках

In [ ]:
def clear_summaries():
    for f in SUMMARY_DIR.glob("summary_batch_*.txt"):
        f.unlink()


clear_summaries()
print("Old summaries cleared.")

### ручной тест

In [18]:
print(search_project_threads.invoke({"project_hint": "Segezha", "limit": 10}))

{
  "project_hint": "Segezha",
  "threads": [
    {
      "thread_key": "Обсуждение вопросов по ЗНИ-282 \"ЗНИ_Изменение схемы учета вывозки сырья с филиалов Лесных ресурсов на СЦБК\"",
      "subject": "RE: Обсуждение вопросов по ЗНИ-282 \"ЗНИ_Изменение схемы учета вывозки сырья с филиалов Лесных ресурсов на СЦБК\"",
      "participants": [
        "IMCEAEX-_o=ilp_ou=Exchange+20Administrative+20Group+20+28FYDIBOHF23SPDLT+29_cn=Recipients_cn=user128488d2@bearingpoint.ru",
        "Khusainov_DS@segezha-group.com",
        "Lytkin_AV@segezha-group.com",
        "Safonov_AY@segezha-group.com",
        "alexey.ivanenko@bearingpoint.ru",
        "andrey.kryukov@bearingpoint.ru",
        "ce-sergey.schepin@bearingpoint.ru",
        "dmitry.kubyshkin@bearingpoint.ru",
        "petr.ostrik@bearingpoint.ru",
        "rasul.abdulin@bearingpoint.ru"
      ],
      "keywords": [
        "Skype",
        "Алексей",
        "Алексей Лыткин",
        "ЗНИ",
        "ЗНИ MNF",
        "ЗНИ с номером",


In [19]:
print(get_project_corpus_batch.invoke({
    "project_hint": "Segezha",
    "offset": 0,
    "batch_size": 5,
    "thread_limit": 10
}))

{
  "project_hint": "Segezha",
  "offset": 0,
  "batch_size": 5,
  "batch_len": 5,
  "has_more": true,
  "thread_keys": [
    "Обсуждение вопросов по ЗНИ-282 \"ЗНИ_Изменение схемы учета вывозки сырья с филиалов Лесных ресурсов на СЦБК\"",
    "Интеграция ТМ с ЭТРАН на пилотной площадке ВФК",
    "Корректность данных в sap для расчета выручки (добавила повестку) ЗНИ 272",
    "ЗНИ по  О2С – Сбыт, продажи.",
    "bw Отчет по Лесообеспечению",
    "Зеркало 2: мемо - для информации по валютным счетам",
    "Сверка ТО"
  ],
  "total_messages": 746,
  "batch": [
    {
      "thread_key": "Интеграция ТМ с ЭТРАН на пилотной площадке ВФК",
      "email_id": "ef1f8452-7806-4b72-80e6-dc3fac2303e7",
      "message_id": "<bd13504b14f44d3c94fc4835eea77762@ex2.be.local>",
      "subject": "FW:  Интеграция ТМ с ЭТРАН на пилотной площадке ВФК",
      "topic": "Интеграция SAP TM с ЭТРАН",
      "sent_at_utc": "2020-11-25",
      "participants": [
        "ce-Alexey.Fedoseev@bearingpoint.ru",
        "ce